# Per-tree instances on 10 cm ortho — DeepForest

- `uv add deepforest`

In [ ]:
# %pip install deepforest
import numpy as np
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

image_path = '/home/ubuntu/work/satellite_data/digital_orthophoto_nrw/2021/2021/dop10rgbi_32_280_5652_1_nw_2021.tif'
WIN = Window(2000, 3000, 1000, 1000)

In [ ]:
def read_rgb(src, window):
    img = np.moveaxis(src.read([1, 2, 3], window=window), 0, -1)
    if img.dtype == np.uint16:            # 8-bit for DeepForest
        img = (img / 256).astype(np.uint8)
    return np.ascontiguousarray(img)

with rasterio.open(image_path) as src:
    crop = read_rgb(src, WIN)
    win_transform = src.window_transform(WIN)  # pixel -> world for this crop
    crs = src.crs                               # native (UTM, metric)
print('crop:', crop.shape, '| crs:', crs)
plt.imshow(crop); plt.axis('off');

In [ ]:
from deepforest import main

m = main.deepforest()
try:
    m.load_model('weecology/deepforest-tree')  # pretrained crown detector (HF)
except Exception:
    m.use_release()  

In [ ]:
# tile the crop internally; patch_size ~40 m at 10 cm is a good crown context
boxes = m.predict_tile(image=crop, patch_size=400, patch_overlap=0.1)
print(len(boxes), 'trees')
boxes.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(crop)
for _, r in boxes.iterrows():
    ax.add_patch(mpatches.Rectangle((r.xmin, r.ymin), r.xmax - r.xmin, r.ymax - r.ymin,
                                    fill=False, edgecolor='lime', linewidth=1))
ax.set_title(f'DeepForest: {len(boxes)} trees'); ax.axis('off');

# To GeoJson

In [ ]:
import geopandas as gpd
from shapely.geometry import box as shp_box
from rasterio.transform import xy

def to_world(col, row):
    x, y = xy(win_transform, row, col, offset='ul')
    return x, y

geoms = []
for _, r in boxes.iterrows():
    x0, y0 = to_world(r.xmin, r.ymin)
    x1, y1 = to_world(r.xmax, r.ymax)
    geoms.append(shp_box(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)))

gdf = gpd.GeoDataFrame({'score': boxes.score.values}, geometry=geoms, crs=crs)
gdf['area_m2'] = gdf.geometry.area          # native CRS is metric
gdf = gdf.to_crs('EPSG:4326')
print(len(gdf), 'features')
gdf.head()